# Problema 1: Distribución de Agua en una Red de Emergencia

Tras un desastre natural, una ciudad ha habilitado una red temporal de distribución de agua potable. El sistema está compuesto por:

- un nodo fuente principal que almacena agua,
- varios centros de distribución intermedios,
- y múltiples zonas afectadas que requieren abastecimiento.

Cada tubería de la red posee una capacidad máxima de flujo. Además, algunas conexiones representan rutas críticas con restricciones especiales de transporte.

El objetivo es determinar el máximo flujo de agua que puede enviarse desde la central principal hacia las zonas de consumo respetando todas las capacidades de la red.

---

## Red de distribución

La red está modelada como un grafo dirigido.

Cada arco posee:

```text
(origen, destino, capacidad)
```

La red disponible es:

```python
arcos = [
    (0,1,15),
    (0,2,10),

    (1,3,8),
    (1,4,7),

    (2,4,10),
    (2,5,5),

    (3,6,10),
    (4,6,5),
    (4,7,7),

    (5,7,8),

    (6,8,10),
    (7,8,10)
]
```

Donde:

- 0 representa la central principal,
- 8 representa la zona destino final,
- los demás nodos son centros intermedios.

---

## Restricciones adicionales

1. El flujo total que atraviesa el nodo 4 no puede superar 10 unidades debido a daños estructurales.

2. Las rutas:
   ```text
   (1 → 3) y (2 → 5)
   ```
   son consideradas rutas prioritarias y deben transportar al menos 3 unidades de flujo cada una.

3. Ningún arco puede superar su capacidad máxima.

---

## Objetivo

Determinar:

- el flujo máximo total desde la fuente hasta el destino,
- el flujo enviado por cada arco,
- y verificar automáticamente todas las restricciones.

---

## Requerimientos

El problema debe resolverse utilizando:

- arcos de flujo,
- capacidades sobre arcos,
- conservación de flujo,
- restricciones locales sobre nodos,
- y restricciones globales sobre la red.

Se debe modelar mediante flujo en redes usando OR-Tools.

In [17]:
from ortools.sat.python import cp_model

In [18]:
model = cp_model.CpModel()

In [19]:
arcos = [
    (0,1,15),
    (0,2,10),

    (1,3,8),
    (1,4,7),

    (2,4,10),
    (2,5,5),

    (3,6,10),
    (4,6,5),
    (4,7,7),

    (5,7,8),

    (6,8,10),
    (7,8,10)
]

source = 0
sink = 8

In [20]:
flujo = {}

for i,j,c in arcos:
    flujo[(i,j)] = model.NewIntVar(0, c, f"f_{i}_{j}")

In [21]:

nodos = set()

for i,j,c in arcos:
    nodos.add(i)
    nodos.add(j)

for nodo in nodos:

    if nodo == source or nodo == sink:
        continue

    entrada = []
    salida = []

    for i,j,c in arcos:

        if j == nodo:
            entrada.append(flujo[(i,j)])

        if i == nodo:
            salida.append(flujo[(i,j)])

    model.Add(sum(entrada) == sum(salida))

In [22]:
# Restriccion local del nodo 4
entrada_4 = []
salida_4 = []

for i,j,c in arcos:

    if j == 4:
        entrada_4.append(flujo[(i,j)])

    if i == 4:
        salida_4.append(flujo[(i,j)])

model.Add(sum(entrada_4) <= 10)
model.Add(sum(salida_4) <= 10)

In [23]:
# Restricciones Globales

# (1 -> 3) al menos 3 unidades
model.Add(flujo[(1,3)] >= 3)

# (2 -> 5) al menos 3 unidades
model.Add(flujo[(2,5)] >= 3)

In [24]:
flujo_total = model.NewIntVar(0, 1000, "flujo_total")

salidas_source = []
entradas_sink = []

for i,j,c in arcos:

    if i == source:
        salidas_source.append(flujo[(i,j)])

    if j == sink:
        entradas_sink.append(flujo[(i,j)])

model.Add(flujo_total == sum(salidas_source))
model.Add(flujo_total == sum(entradas_sink))

In [25]:
model.Maximize(flujo_total)

solver = cp_model.CpSolver()

status = solver.Solve(model)

In [26]:
if status == cp_model.OPTIMAL:

    print("="*25)
    print(" FLUJO MÁXIMO ENCONTRADO")
    print("="*25)

    print(f"\nFlujo total enviado: {solver.Value(flujo_total)}\n")

    nombres = {
    0: "Central 0",
    1: "Central 1",
    2: "Central 2",
    3: "Central 3",
    4: "Central 4",
    5: "Central 5",
    6: "Central 6",
    7: "Central 7",
    8: "Central 8"
    }

    print("Flujo por arco:\n")

    for i,j,c in arcos:

        valor = solver.Value(flujo[(i,j)])

        print(
            f"{nombres[i]}  -->  {nombres[j]}"
            f"\n   Flujo enviado: {valor}"
            f"\n   Capacidad máxima: {c}\n"
        )

    print("\n" + "="*32)
    print(" VERIFICACIÓN DE RESTRICCIONES")
    print("="*32)

    # Nodo 4
    flujo_nodo_4 = sum(
        solver.Value(flujo[(i,j)])
        for i,j,c in arcos
        if j == 4
    )

    print(f"\nFlujo atravesando nodo 4: {flujo_nodo_4} <= 10")

    # Rutas prioritarias
    print(
        f"\nFlujo en (1->3): {solver.Value(flujo[(1,3)])} >= 3"
    )

    print(
        f"\nFlujo en (2->5): {solver.Value(flujo[(2,5)])} >= 3"
    )

else:
    print("No se encontró solución factible")

 FLUJO MÁXIMO ENCONTRADO

Flujo total enviado: 20

Flujo por arco:

Central 0  -->  Central 1
   Flujo enviado: 10
   Capacidad máxima: 15

Central 0  -->  Central 2
   Flujo enviado: 10
   Capacidad máxima: 10

Central 1  -->  Central 3
   Flujo enviado: 8
   Capacidad máxima: 8

Central 1  -->  Central 4
   Flujo enviado: 2
   Capacidad máxima: 7

Central 2  -->  Central 4
   Flujo enviado: 5
   Capacidad máxima: 10

Central 2  -->  Central 5
   Flujo enviado: 5
   Capacidad máxima: 5

Central 3  -->  Central 6
   Flujo enviado: 8
   Capacidad máxima: 10

Central 4  -->  Central 6
   Flujo enviado: 2
   Capacidad máxima: 5

Central 4  -->  Central 7
   Flujo enviado: 5
   Capacidad máxima: 7

Central 5  -->  Central 7
   Flujo enviado: 5
   Capacidad máxima: 8

Central 6  -->  Central 8
   Flujo enviado: 10
   Capacidad máxima: 10

Central 7  -->  Central 8
   Flujo enviado: 10
   Capacidad máxima: 10


 VERIFICACIÓN DE RESTRICCIONES

Flujo atravesando nodo 4: 7 <= 10

Flujo en (1->3

# Problema 2: Asignación Óptima de Drones de Emergencia

Una ciudad inteligente ha desplegado múltiples drones de respuesta rápida para atender emergencias urbanas.

Cada dron puede cubrir únicamente ciertos distritos debido a restricciones de autonomía, clima y rutas aéreas.

Además:

- cada dron solo puede ser asignado a un distrito,
- cada distrito puede recibir como máximo un dron.

Sin embargo, la ciudad ha añadido restricciones operativas globales:

1. Algunos distritos son críticos y deben recibir cobertura obligatoria.
2. Ciertas zonas aéreas tienen capacidad limitada y solo permiten un número máximo de drones simultáneos.
3. Algunos drones especializados poseen prioridad operativa y deben ser utilizados obligatoriamente.

El objetivo es determinar la asignación óptima de drones maximizando la cobertura total y respetando todas las restricciones.

---

# Entrada

La primera línea contiene:

```text
D R
```

donde:

- D = número de distritos,
- R = número de drones.

Luego siguen R líneas:

```text
k d1 d2 d3 ...
```

donde:

- k = cantidad de distritos alcanzables,
- d_i = identificador del distrito.

---

# Restricciones

## Distritos críticos

Los distritos:

```text
3 y 7
```

deben recibir obligatoriamente un dron.

---

## Restricción aérea global

Los distritos:

```text
4, 5 y 6
```

pertenecen al mismo corredor aéreo y pueden recibir como máximo:

```text
2 drones simultáneamente
```

---

## Drones prioritarios

Los drones:

```text
0 y 3
```

deben ser utilizados obligatoriamente.

---

# Objetivo

Determinar:

- el máximo número de distritos cubiertos,
- las asignaciones dron → distrito,
- y verificar automáticamente todas las restricciones.

In [27]:
from collections import deque
from ortools.sat.python import cp_model

In [28]:
class HopcroftKarp:

    def __init__(self, left_size, right_size, adjacency):

        self.left_size = left_size
        self.right_size = right_size
        self.adj = adjacency

        self.pair_u = [-1] * left_size
        self.pair_v = [-1] * right_size
        self.dist = [0] * left_size

    def bfs(self):

        queue = deque()

        for u in range(self.left_size):

            if self.pair_u[u] == -1:
                self.dist[u] = 0
                queue.append(u)
            else:
                self.dist[u] = float("inf")

        found = False

        while queue:

            u = queue.popleft()

            for v in self.adj[u]:

                pu = self.pair_v[v]

                if pu != -1 and self.dist[pu] == float("inf"):

                    self.dist[pu] = self.dist[u] + 1
                    queue.append(pu)

                if pu == -1:
                    found = True

        return found

    def dfs(self, u):

        for v in self.adj[u]:

            pu = self.pair_v[v]

            if pu == -1 or (
                self.dist[pu] == self.dist[u] + 1
                and self.dfs(pu)
            ):

                self.pair_u[u] = v
                self.pair_v[v] = u

                return True

        self.dist[u] = float("inf")

        return False

    def max_matching(self):

        matching = 0

        while self.bfs():

            for u in range(self.left_size):

                if self.pair_u[u] == -1:

                    if self.dfs(u):
                        matching += 1

        return matching

In [29]:
INPUT_DATA = '''8 8
3 0 1 2
3 1 3 4
2 2 5
4 0 4 5 6
3 3 6 7
2 1 7
3 2 4 6
2 5 7'''

lines = INPUT_DATA.strip().splitlines()

data = []

for line in lines:
    data.extend(map(int, line.split()))

D, R = data[0], data[1]

idx = 2

adjacency = []

for _ in range(R):

    k = data[idx]
    idx += 1

    districts = data[idx:idx+k]
    idx += k

    adjacency.append(districts)

In [30]:
hk = HopcroftKarp(
    left_size=R,
    right_size=D,
    adjacency=adjacency
)

matching = hk.max_matching()

print("Matching máximo inicial:", matching)

print("\nAsignaciones HK:\n")

for drone, district in enumerate(hk.pair_u):

    if district != -1:
        print(f"Dron {drone} -> Distrito {district}")

Matching máximo inicial: 8

Asignaciones HK:

Dron 0 -> Distrito 0
Dron 1 -> Distrito 1
Dron 2 -> Distrito 2
Dron 3 -> Distrito 4
Dron 4 -> Distrito 3
Dron 5 -> Distrito 7
Dron 6 -> Distrito 6
Dron 7 -> Distrito 5


In [31]:
model = cp_model.CpModel()

x = {}

for drone in range(R):

    for district in adjacency[drone]:

        x[(drone, district)] = model.NewBoolVar(
            f"x_{drone}_{district}"
        )

In [ ]:
# Cada dron máximo un distrito

for drone in range(R):

    vars_drone = []

    for district in adjacency[drone]:

        vars_drone.append(x[(drone, district)])

    model.Add(sum(vars_drone) <= 1)

In [ ]:
# Cada distrito máximo un dron

for district in range(D):

    vars_district = []

    for drone in range(R):

        if district in adjacency[drone]:

            vars_district.append(x[(drone, district)])

    model.Add(sum(vars_district) <= 1)

In [ ]:
# Distritos críticos obligatorios

critical = [3, 7]

for district in critical:

    vars_district = []

    for drone in range(R):

        if district in adjacency[drone]:

            vars_district.append(x[(drone, district)])

    model.Add(sum(vars_district) >= 1)

In [ ]:
# Corridor aéreo global

corridor = [4,5,6]

corridor_vars = []

for drone in range(R):

    for district in corridor:

        if district in adjacency[drone]:

            corridor_vars.append(x[(drone, district)])

model.Add(sum(corridor_vars) <= 2)

In [ ]:
# Drones prioritarios

priority_drones = [0,3]

for drone in priority_drones:

    vars_drone = []

    for district in adjacency[drone]:

        vars_drone.append(x[(drone, district)])

    model.Add(sum(vars_drone) == 1)

In [ ]:
# Maximizar cobertura

model.Maximize(sum(x.values()))

solver = cp_model.CpSolver()

status = solver.Solve(model)

In [40]:
if status == cp_model.OPTIMAL:

    print("\nCobertura óptima:", int(solver.ObjectiveValue()))

    print("\nAsignaciones finales:\n")

    for (drone, district), var in x.items():

        if solver.Value(var):

            print(f"Dron {drone} -> Distrito {district}")


Cobertura óptima: 7

Asignaciones finales:

Dron 0 -> Distrito 2
Dron 1 -> Distrito 3
Dron 3 -> Distrito 0
Dron 4 -> Distrito 7
Dron 5 -> Distrito 1
Dron 6 -> Distrito 6
Dron 7 -> Distrito 5
